# SearchLite-RL FINAL FIXES: verify, recover, regenerate, package (Colab, CPU only)
Nothing here trains, retrains or re-evaluates a model, changes a prompt, or touches the frozen checkpoint. Google Drive is **read-only** except for **new** files written under new names (`submission_package_v2*`); an existing package is never overwritten.
Run the cells in order. Runtime type: CPU is enough (a GPU is not used).

0. Push the new commit first (done on your Mac), then run cell 2 so Colab has it (cells are numbered from the first code cell = 1).
1. Cell 4 **checks your reported sealed numbers against the raw JSONs, prints the paired comparisons, and verifies the freeze** and that no protected code changed since the freeze commit.
2. Cell 5 **finds the original GRPO training log** on Drive (copies only `train_metrics.csv` if it exists; otherwise the figure is reported as unavailable with the reason).
3. Cell 6 (optional) uploads your executed notebooks so they are packaged.
4. Cell 7 regenerates tables, figures, case studies and the report and builds a **versioned** zip. Cell 8 verifies the zip. Cell 9 copies to Drive without overwriting and compares checksums with the earlier package.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/headroom-grpo && git pull -q; else git clone -q -b experiment/headroom-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
cd /content/Pilot-project
python -c "import trl,peft,transformers,matplotlib,pandas" 2>/dev/null && echo "dependencies already installed" || { pip install -q -r requirements.txt; echo "INSTALLED: restart the runtime and re-run this cell"; exit 0; }
git log --oneline | head -3
test -f src/two_comparisons.py && echo "new fix modules present" || { echo "MISSING src/two_comparisons.py: push the commit first"; exit 1; }

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p outputs
for s in study_sft study_hard study_early study_rl_only; do [ -d $D/$s ] && ln -sfn $D/$s outputs/$s && echo "linked $s" || echo "not on Drive: $s"; done
mkdir -p outputs/main_v2
[ -f outputs/main_v2/adapter/adapter_config.json ] || cp -rn $D/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn $D/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "condition A adapter not found"
echo "--- previous package on Drive (read-only look) ---"
ls -la $D/submission_package 2>/dev/null | head -20 || echo "no submission_package folder"
ls $D/*.zip 2>/dev/null || true

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
S=outputs/study_hard
echo "### 1. Reported sealed numbers vs raw evaluation JSONs, and the two paired comparisons (stdout only; nothing is written to Drive)"
python -m src.two_comparisons --study $S --split hard_final --headline M3_s42 --replications M3_s43 \
   --expect M2=34/120 M2_typed=75/120 M3_s42=82/120 M3_s43=81/120 M3_s42:reverse_two_hop=4/20 M3_s42:unanswerable=0/20
echo
echo "### 2. Freeze integrity"
python -m src.verify_freeze --study $S

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# Looks for the ORIGINAL (from-base) GRPO run's train_metrics.csv on Drive. Copies it only if absent here; never invents or estimates anything.
mkdir -p submission/tables
python -m src.recover_train_logs --run outputs/main_v2 --search /content/drive/MyDrive/SearchLite-RL --report submission/tables/original_grpo_log_recovery.md
echo; echo "--- every train_metrics.csv on Drive (for your own check) ---"
find /content/drive/MyDrive/SearchLite-RL -name train_metrics.csv -maxdepth 4 2>/dev/null | sed 's#/content/drive/MyDrive/SearchLite-RL/##' | sort | head -40

In [ ]:
# OPTIONAL: upload the executed notebooks (File > Download .ipynb in each Colab session, or from ~/Downloads) so they are packaged with their outputs.
import os
from google.colab import files
os.makedirs('/content/Pilot-project/executed_notebooks', exist_ok=True)
up = files.upload()
for name, data in up.items():
    dst = os.path.join('/content/Pilot-project/executed_notebooks', name)
    if os.path.exists(dst):
        print('kept existing', name)
    else:
        open(dst, 'wb').write(data); print('saved', name, len(data), 'bytes')

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
rm -rf submission     # the local working copy only; archived zips and everything on Drive are untouched
MODE=post VERSION=v2_$(date +%Y%m%d_%H%M) bash run_final_analysis.sh 2>&1 | tail -120
echo; cat submission/STATUS_CHECKLIST.md
ls -la ../Pilot-project/*.zip ../Pilot-project/*.sha256

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
Z=$(ls -t searchlite_rl_submission_v2_*.zip | head -1)
echo "zip: $Z"; sha256sum "$Z"; cat "$Z.sha256"
python - "$Z" <<'PY'
import sys, zipfile, hashlib, re
z = zipfile.ZipFile(sys.argv[1]); names = z.namelist()
print(len(names), "files; bad entry:", z.testzip())
for top in ("FINAL_PILOT_REPORT.md","README_SUBMISSION.md","STATUS_CHECKLIST.md","FROZEN_HARD.json","case_studies.md","MANIFEST_SHA256.txt"):
    print("  ", "OK  " if top in names else "MISSING", top)
for pre in ("figures/","tables/","raw/","code/src/","code/configs/","code/data/","notebooks/","checkpoint/frozen_headline_adapter/"):
    print("  ", f"{sum(n.startswith(pre) for n in names):4d}", pre)
rep = z.read("FINAL_PILOT_REPORT.md").decode()
print("NOT AVAILABLE blocks:", len(re.findall(r"^> \*\*NOT AVAILABLE:\*\*", rep, re.M)), "| placeholders:", rep.count("{{"))
bad = [n for n in names if re.search(r"(^|/)\.env|\.pem$|\.key$|__pycache__|\.pt$|\.bin$", n)]
print("forbidden files in zip:", bad or "none")
man = dict(l.split("  ", 1)[::-1] for l in z.read("MANIFEST_SHA256.txt").decode().splitlines())
wrong = [n for n, h in man.items() if hashlib.sha256(z.read(n)).hexdigest() != h]
print("manifest hashes verified for", len(man), "files; mismatches:", wrong or "none")
PY

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
Z=$(ls -t searchlite_rl_submission_v2_*.zip | head -1)
mkdir -p $D/submission_package_v2
# never overwrite: refuse if a file with the same name but different bytes already exists
for f in "$Z" "$Z.sha256"; do
  if [ -e "$D/submission_package_v2/$f" ]; then
    cmp -s "$f" "$D/submission_package_v2/$f" && echo "already on Drive, identical: $f" || { echo "REFUSING: $f exists on Drive with different content"; exit 1; }
  else cp "$f" "$D/submission_package_v2/$f" && echo "copied $f"; fi
done
cp -n -r submission $D/submission_package_v2/unzipped 2>/dev/null || echo "unzipped copy already on Drive (kept)"
echo "--- checksums: local vs Drive (v2) ---"
sha256sum "$Z" "$D/submission_package_v2/$Z"
echo "--- earlier package on Drive (read-only comparison) ---"
for old in $D/submission_package/searchlite_rl_submission*.zip $D/searchlite_rl_submission*.zip; do
  [ -f "$old" ] && { sha256sum "$old"; python - "$old" "$Z" <<'PY'
import sys, zipfile
old, new = zipfile.ZipFile(sys.argv[1]), zipfile.ZipFile(sys.argv[2])
print("earlier zip:", len(old.namelist()), "files | new zip:", len(new.namelist()), "files")
print("earlier report identical to new report:", old.read("FINAL_PILOT_REPORT.md") == new.read("FINAL_PILOT_REPORT.md") if "FINAL_PILOT_REPORT.md" in old.namelist() else "n/a")
print("only in new:", sorted(set(new.namelist()) - set(old.namelist()))[:12], "...")
PY
  }
done || true